# Telco Churn Final Runbook

This notebook uses the cleaned project structure and runs only the final workflow.

Workflow:
1. Check and install required packages.
2. Validate required project files.
3. Run model_search_ensemble.py (final model pipeline).
4. Run generate_result_graphs.py to rebuild charts.
5. Display final deliverables and key metrics.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "scikit-learn",
    "xgboost",
    "shap",
    "optuna",
    "lightgbm",
]

missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required packages are already installed.")

Installing missing packages: ['scikit-learn', 'catboost']


In [6]:
from pathlib import Path

candidates = [
    Path.cwd(),
    Path("/Users/sreeharshak/Dev/Customer Segmentation & Retention"),
]

PROJECT_DIR = None
for cand in candidates:
    if (cand / "model_search_ensemble.py").exists():
        PROJECT_DIR = cand
        break

if PROJECT_DIR is None:
    for cand in Path.home().glob("Dev/*"):
        if (cand / "model_search_ensemble.py").exists():
            PROJECT_DIR = cand
            break

if PROJECT_DIR is None:
    PROJECT_DIR = Path.cwd()
    print("Warning: could not auto-detect project directory. Using current working directory:", PROJECT_DIR)

required_files = [
    "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    "model_search_ensemble.py",
    "generate_result_graphs.py",
    "PROJECT_SUMMARY.md",
    "RESULTS_ONE_PAGE.md",
    "recommendation.txt",
]

missing_files = [f for f in required_files if not (PROJECT_DIR / f).exists()]
if missing_files:
    print("Warning: missing files from detected project dir:", missing_files)
else:
    print("All required files are present in:", PROJECT_DIR)

## Run Final Model Pipeline

In [ ]:
import subprocess
import sys

script = PROJECT_DIR / "model_search_ensemble.py"
if not script.exists():
    print("Cannot run model pipeline. Missing:", script)
else:
    result = subprocess.run(
        [sys.executable, "model_search_ensemble.py"],
        cwd=str(PROJECT_DIR),
        check=True,
        capture_output=True,
        text=True,
    )
    lines = result.stdout.strip().splitlines()
    print("\n".join(lines[-120:]))

## Rebuild Graphs

In [ ]:
import subprocess
import sys

script = PROJECT_DIR / "generate_result_graphs.py"
if not script.exists():
    print("Cannot rebuild graphs. Missing:", script)
else:
    graphs = subprocess.run(
        [sys.executable, "generate_result_graphs.py"],
        cwd=str(PROJECT_DIR),
        check=True,
        capture_output=True,
        text=True,
    )
    print(graphs.stdout)

## Display Final Plots

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

image_files = [
    "prob_distribution.png",
    "Results/results_metrics_comparison.png",
    "Results/results_model_summary.png",
    "Results/results_business_cost_breakdown.png",
]

for img_file in image_files:
    p = PROJECT_DIR / img_file
    if p.exists():
        img = mpimg.imread(p)
        plt.figure(figsize=(11, 5))
        plt.imshow(img)
        plt.title(img_file)
        plt.axis("off")
        plt.show()
    else:
        print(f"Missing plot: {img_file}. Run previous cells first.")

## Key Metrics Snapshot

In [ ]:
import json
from pathlib import Path

report_path = PROJECT_DIR / "ensemble_optimization_report.json"
if not report_path.exists():
    print("Cannot show metrics. Missing:", report_path)
else:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    op = report["recommended_operating_point"]

    print("Selected final model:", report.get("selected_final_model"))
    print("Recommended threshold:", round(op["threshold"], 2))
    print("ROC-AUC:", round(op["roc_auc"], 4))
    print("Precision:", round(op["precision"], 4))
    print("Recall:", round(op["recall"], 4))
    print("F1:", round(op["f1"], 4))
    print("Accuracy:", round(op["accuracy"], 4))

    print("\nEngineered features in top-20:")
    for f in report.get("engineered_features_in_top_20", []):
        print("-", f)

## Final Text Deliverables

- recommendation.txt contains the business action plan with exact segment counts, churn rates, threshold, and cost mapping.
- RESULTS_ONE_PAGE.md contains the one-page model card, segment table, churn drivers, and budget/ROI summary.

Run the next cell to print both documents in full.

In [ ]:
from pathlib import Path

for doc in ["recommendation.txt", "RESULTS_ONE_PAGE.md", "PROJECT_SUMMARY.md"]:
    p = Path(doc)
    print("\n" + "=" * 88)
    print(doc)
    print("=" * 88)
    if p.exists():
        print(p.read_text(encoding="utf-8"))
    else:
        print(f"Missing file: {doc}")